In [1]:
pip install qiskit

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ------------- -------------------------- 3.1/9.3 MB 22.7 MB/s eta 0:00:01
   ------------------ --------------------- 4.2/9.3 MB 16.0 MB/s eta 0:00:01
   ------------------------ --------------- 5.8/9.3 MB 9.7 MB/s eta 0:00:01
   --------------------------- ------------ 6.3/9.3 MB 8.6 MB/s eta 0:00:01
   ----------------------------- ---------- 6.8/9.3 MB 6.6 MB/s eta 0:00:01
   ------------------------------ --------- 7.1/9.3 MB 5.9 MB/s eta 0:00:01
   -------------------------------- ------- 7.6/9.3 MB 5.4 MB/s eta 0:00:01
   ------------------------------------ --- 8.4/9.3 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------  9.2/9.3 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 4.7 MB/s  0:00:01
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ------------------ ---

In [1]:
pip install qiskit-nature[pyscf] -U

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   -------------------------------------- - 2.1/2.2 MB 27.5 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 8.7 MB/s  0:00:00

   ---------------------------------------- 0/2 [qiskit-algorithms]
   ---------------------------------------- 0/2 [qiskit-algorithms]
   ---------------------------------------- 0/2 [qiskit-algorithms]
   ---------------------------------------- 0/2 [qiskit-algorithms]
   ---------------------------------------- 0/2 [qiskit-algorithms]
   ---------------------------------------- 0/2 [qiskit-algorithms]
   -------------------- ------------------- 1/2 [qiskit-nature]
   -------------------- ------------------- 1/2 [qiskit-nature]
   -------------------- ------------------- 1/2 [qiskit-nature]
   -------------------- ------------------- 1/2 [qiskit-nature]
   -------------------- --------------

In [3]:
#from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper

ImportError: DLL load failed while importing _accelerate: An Application Control policy has blocked this file.

In [ ]:
# Define a fermionic operator
fermionic_operator = FermionicOp(
    {"+_0 -_1": 1.0, "+_1 -_0": 1.0},
    num_spin_orbitals=2
)

# Map the fermionic operator to qubits
mapper = JordanWignerMapper()
qubit_operator = mapper.map(fermionic_operator)

print(qubit_operator)

In [ ]:
import numpy
import h5py
import qiskit
import qiskit_nature
import qiskit_algorithms

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.units import DistanceUnit
import numpy as np

from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP,CG,SPSA,AQGD
from qiskit.primitives import StatevectorEstimator

from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

# Define the mapper
mapper = JordanWignerMapper()


distances = []
vqe_slsqp_energies = []
vqe_cg_energies = []
vqe_spsa_energies = []
vqe_aqgd_energies = []
exact_energy = []

# Configure VQE
estimator = StatevectorEstimator()
optimizer = SLSQP(maxiter=200)
opt_1 = CG(maxiter=200)
opt_2 = SPSA(maxiter=200)
opt_3 = AQGD(maxiter=200)

for coordinate in np.arange(0.3, 4.0, 0.05):
    dist = "H 0 0 0; H 0 0 {}"

    driver = PySCFDriver(
        atom= dist.format(coordinate),
        basis="sto3g",
        charge=0,
        spin=0,
        unit=DistanceUnit.ANGSTROM,
    )
    problem = driver.run()

    
    initial_state = HartreeFock(
    problem.num_spatial_orbitals,
    problem.num_particles,
    mapper,
    )
    
    
    ansatz = UCCSD(
    problem.num_spatial_orbitals,
    problem.num_particles,
    mapper,
    initial_state=initial_state,
    )
    
    vqe_solver_slsqp = VQE(estimator, ansatz, optimizer)
    calc = GroundStateEigensolver(mapper, vqe_solver_slsqp)
    res = calc.solve(problem)
    vqe_slsqp_energies.append(res.total_energies[0])
    
    vqe_solver_cg = VQE(estimator, ansatz, opt_1)
    calc = GroundStateEigensolver(mapper, vqe_solver_cg)
    res = calc.solve(problem)
    vqe_cg_energies.append(res.total_energies[0])
    
    vqe_solver_spsa = VQE(estimator, ansatz, opt_2)
    calc = GroundStateEigensolver(mapper, vqe_solver_spsa)
    res = calc.solve(problem)
    vqe_spsa_energies.append(res.total_energies[0])
    
    vqe_solver_aqgd = VQE(estimator, ansatz, opt_3)
    calc = GroundStateEigensolver(mapper, vqe_solver_aqgd)
    res = calc.solve(problem)
    vqe_aqgd_energies.append(res.total_energies[0])
    #distances.append(coordinate)
    
    
    solver = NumPyMinimumEigensolver()
    calc = GroundStateEigensolver(mapper, solver)
    res_exact = calc.solve(problem)
    exact_energy.append(res_exact.total_energies[0])
    
    
    distances.append(coordinate)


print("Number of distances:", len(distances))
print("Number of exact energies:", len(exact_energy))
print("Number of VQE energies:", len(vqe_slsqp_energies))
print("Number of VQE1 energies:", len(vqe_cg_energies))
print("Number of VQE2 energies:", len(vqe_spsa_energies))
print("Number of VQE3 energies:", len(vqe_aqgd_energies))
#print("Distances:", distances)
#print("Exact energies:", exact_energy)

#print("Distances:", distances)
#print("Exact energies:", exact_energy)

In [ ]:
# Exact solution
i = np.argmin(exact_energy)
print(f"Exact: Energy = {exact_energy[i]:.12f} Hartree, "
      f"Distance = {distances[i]:.4f} Å")

In [ ]:
# VQE-CG
i = np.argmin(vqe_cg_energies)
print(f"VQE-CG: Energy = {vqe_cg_energies[i]:.12f} Hartree, "
      f"Distance = {distances[i]:.4f} Å")